In [ ]:
from model.attention import FlashGroupQueryAttention, GroupQueryAttention, compute_rope_params
import torch
import time
# from model.utils import co


In [ ]:
from model.Qwen3 import Qwen3Model


In [ ]:
d_model=24
g=torch.Generator().manual_seed(123)
torch.manual_seed(123)
cfg={}
import torch
batch_size=4
seq_len=4096
emb_dim=512
cfg['emb_dim']=emb_dim
cfg['dtype']=torch.float32
cfg['vocab_size']=132000
cfg['num_heads']=4
cfg["num_kv_groups"]=2
cfg["head_dim"]=cfg['emb_dim']//cfg['num_heads']
B=torch.randn(batch_size, seq_len,emb_dim, dtype=cfg['dtype'], generator=g)
gq=GroupQueryAttention(d_in=cfg["emb_dim"],num_kv_groups=cfg["num_kv_groups"], num_heads=cfg["num_heads"],head_dim=cfg["head_dim"], dtype=cfg["dtype"])
start_pos=0
end_pos=start_pos+B.shape[1]
mask=torch.triu(
    torch.ones(end_pos,end_pos,device=B.device, dtype=torch.bool), diagonal=1
)[start_pos:end_pos, :end_pos]
print(mask)
print(gq)
start=time.time()
cos, sin=compute_rope_params(head_dim=cfg["head_dim"],context_length=seq_len)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
B = B.to(device)
gq = gq.to(device)
cos = cos.to(device)
sin = sin.to(device)
mask = mask.to(device)
print(gq(B, mask,cos, sin, start_pos=start_pos))
print("logger timer : ", time.time()-start)

In [ ]:

cfg["vocab_size"]= 151936

cfg["n_layers"] =1 #28
# hidden_dim : 3072
# head_dim : 128
# qk_norm : True
cfg["n_kv_groups"] = 2
cfg['n_heads']=4
cfg['qk_norm']=True
cfg['hidden_dim']=4*cfg['emb_dim']
cfg["rope_base"] = 1000000
# dtype : bfloat16
cfg["context_length"] =512 #2048
model=Qwen3Model(cfg)
model

In [ ]:
for pn,p in model.named_parameters() : 
    print(pn)

In [ ]:
import torch.nn as nn
from model.attention import RMSNorm , apply_rope
from torch.nn.attention import sdpa_kernel, SDPBackend
class FlashGroupQueryAttention(nn.Module) : 
    def __init__(self, d_in,num_kv_groups, num_heads=None,head_dim=None,qk_norm=False,dtype=None):

        super().__init__()
        assert num_heads%num_kv_groups==0
        if head_dim==None: 
            
            assert d_in%num_heads==0 , "The dimension should be a multiple of the number of heads"
            head_dim=d_in//num_heads
        self.head_dim=head_dim
        self.num_heads=num_heads
        self.num_kv_groups=num_kv_groups
        self.group_size=num_heads//num_kv_groups
        self.d_out=num_heads*head_dim

        self.d_in=d_in
        
        self.W_query=nn.Linear(d_in , self.d_out , bias=False, dtype=dtype)
        self.W_key=nn.Linear(d_in, num_kv_groups*self.head_dim, bias=False, dtype=dtype)
        self.W_value=nn.Linear(d_in , num_kv_groups*self.head_dim, bias=False , dtype=dtype)

        self.out_proj=nn.Linear( self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm : 
            self.q_norm=RMSNorm(head_dim, eps=1e-6)
            self.k_norm=RMSNorm(head_dim, eps=1e-6)
        else : 
            self.q_norm=self.k_norm=None

    def forward (self, x,mask,cos, sin, start_pos, cache=None):
        
        batch_size, seq_len, _ =x.shape
        queries=self.W_query(x).view(batch_size,seq_len,self.num_heads, self.head_dim ).transpose(1,2)
        keys_new=self.W_key(x).view(batch_size,seq_len,self.num_kv_groups, self.head_dim ).transpose(1,2)
        values_new=self.W_value(x).view(batch_size,seq_len,self.num_kv_groups, self.head_dim ).transpose(1,2)

        # if norm 
        if self.q_norm : 
            queries=self.q_norm(queries)
        if self.k_norm :
            keys_new=self.k_norm(keys_new)

        #applys the rope on keys and queries
        queries=apply_rope(queries,cos, sin,start_pos)
        keys_new=apply_rope(keys_new, cos, sin,start_pos)

        #build group queries
        # print("logg group size", self.group_size)
        # print("logg  : n_heads", self.num_heads)
        keys_new=keys_new.repeat_interleave(self.group_size, dim=1)
        values_new=values_new.repeat_interleave(self.group_size, dim=1)

        #cache 
        if cache : 
            prev_key , prev_val=cache
            keys=torch.cat([prev_key, keys_new], dim=2)
            values=torch.cat([prev_val, values_new] , dim=2)
        else :
            start_pos=0 
            keys=keys_new
            values=values_new
        next_cache=(keys, values)
        print("q", queries.shape, queries.dtype, queries.device, queries.is_contiguous())
        print("k", keys.shape, keys.dtype, keys.device, keys.is_contiguous())
        print("v", values.shape, values.dtype, values.device, values.is_contiguous())
        print("head_dim", self.head_dim)
        print("flash enabled", torch.backends.cuda.flash_sdp_enabled())
        print("mem efficient enabled", torch.backends.cuda.mem_efficient_sdp_enabled())
        print("math enabled", torch.backends.cuda.math_sdp_enabled())
        print("cuda capability", torch.cuda.get_device_capability())

        with sdpa_kernel(SDPBackend.MATH) : 
            context=torch.nn.functional.scaled_dot_product_attention(
                queries.to(),keys,values, 
                # attn_mask=~mask,
                dropout_p=0.0, 
                is_causal=True,
                # enable_gqa=True, 
                scale=self.head_dim**(-0.5)
            )
        # assert torch.isfinite(context).all(), "context bad after FA"
        context=context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_out)

        return self.out_proj(context) , next_cache



In [ ]:
d_model=24
g=torch.Generator().manual_seed(123)
torch.manual_seed(123)
cfg={}
import torch
batch_size=4
seq_len=4096
emb_dim=512
cfg['emb_dim']=emb_dim
cfg['dtype']=torch.float32
cfg['num_heads']=4
cfg["num_kv_groups"]=2
cfg["head_dim"]=cfg['emb_dim']//cfg['num_heads']
B=torch.randn(batch_size, seq_len,emb_dim, dtype=cfg['dtype'], generator=g)
gq=FlashGroupQueryAttention(d_in=cfg["emb_dim"],num_kv_groups=cfg["num_kv_groups"], num_heads=cfg["num_heads"],head_dim=cfg["head_dim"], dtype=cfg["dtype"])
start_pos=0
end_pos=start_pos+B.shape[1]
mask=torch.triu(
    torch.ones(end_pos,end_pos,device=B.device, dtype=torch.bool), diagonal=1
)[start_pos:end_pos, :end_pos]
print(mask)
print(gq)
start=time.time()
cos, sin=compute_rope_params(head_dim=cfg["head_dim"],context_length=seq_len)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
B = B.to(device)
gq = gq.to(device)
cos = cos.to(device)
sin = sin.to(device)
mask = mask.to(device)
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    out = gq(B, mask, cos, sin, start_pos=start_pos)
print(out)
# print(gq(B, mask,cos, sin, start_pos=start_pos))
print("logger timer : ", time.time()-start)

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_capability())
print("flash enabled:", torch.backends.cuda.flash_sdp_enabled())
print("mem efficient enabled:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("math enabled:", torch.backends.cuda.math_sdp_enabled())
